In [3]:
import pandas as pd
import numpy as np
import sqlite3
from sklearn.impute import KNNImputer

In [4]:
db_path = "../data/pancreas_proteomics.db"
with sqlite3.connect(db_path) as conn:
    df = pd.read_sql("SELECT [Protein ID], [Sample ID], Condition, Intensity FROM expression", conn)
print(df)

        Protein ID  Sample ID Condition    Intensity
0           A6NMZ7  sample_01       T1D    153502.56
1           P02452  sample_01       T1D  98558584.00
2           P02458  sample_01       T1D   3466728.80
3           P02461  sample_01       T1D  70546624.00
4           P02462  sample_01       T1D  29273744.00
...            ...        ...       ...          ...
133573      Q9Y6X8  sample_20   Control          NaN
133574      Q9Y6X9  sample_20   Control     76899.05
133575      Q9Y6Y0  sample_20   Control          NaN
133576      Q9Y6Y8  sample_20   Control    364424.30
133577  A0A0B4J1V0  sample_20   Control          NaN

[133578 rows x 4 columns]


In [5]:
df_pivot = df.pivot(index='Protein ID', columns='Sample ID', values='Intensity')
print(df_pivot)

Sample ID    sample_01  sample_02   sample_03   sample_05  sample_06  \
Protein ID                                                             
A0A075B6K5         NaN        NaN         NaN         NaN        NaN   
A0A075B6R9         NaN        NaN         NaN         NaN        NaN   
A0A075B6S5         NaN        NaN         NaN         NaN        NaN   
A0A096LP01         NaN        NaN         NaN         NaN        NaN   
A0A0A0MS15         NaN  155922.72         NaN         NaN        NaN   
...                ...        ...         ...         ...        ...   
Q9Y6X5      172245.580   75492.77  234866.640  236185.100  116165.43   
Q9Y6X8             NaN        NaN   75900.516   73050.945        NaN   
Q9Y6X9       84686.875   77121.77   83860.055   73670.620   75815.50   
Q9Y6Y0             NaN        NaN         NaN         NaN        NaN   
Q9Y6Y8      286455.030  424791.00  327730.560  288625.660  349400.90   

Sample ID   sample_07   sample_08  sample_09   sample_10  sampl

In [6]:
condition_df = df[['Sample ID', 'Condition']].drop_duplicates()
condition_map = dict(zip(condition_df['Sample ID'], condition_df['Condition']))
print(condition_map)

{'sample_01': 'T1D', 'sample_02': 'T1D', 'sample_03': 'T1D', 'sample_05': 'T1D', 'sample_06': 'T1D', 'sample_07': 'T1D', 'sample_08': 'T1D', 'sample_09': 'T1D', 'sample_10': 'T1D', 'sample_11': 'Control', 'sample_12': 'Control', 'sample_13': 'Control', 'sample_15': 'Control', 'sample_16': 'Control', 'sample_17': 'Control', 'sample_18': 'Control', 'sample_19': 'Control', 'sample_20': 'Control'}


In [7]:
t1d_id = [id for id, cond in condition_map.items() if cond == 'T1D']
ctrl_id = [id for id, cond in condition_map.items() if cond == 'Control']
#t1d_id, ctrl_id

df_pivot['T1D_count'] = df_pivot[t1d_id].notna().sum(axis=1)
df_pivot['Ctrl_count'] = df_pivot[ctrl_id].notna().sum(axis=1)
df_pivot

Sample ID,sample_01,sample_02,sample_03,sample_05,sample_06,sample_07,sample_08,sample_09,sample_10,sample_11,sample_12,sample_13,sample_15,sample_16,sample_17,sample_18,sample_19,sample_20,T1D_count,Ctrl_count
Protein ID,,,,,,,,,,,,,,,,,,,,
A0A075B6K5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
A0A075B6R9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
A0A075B6S5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
A0A096LP01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
A0A0A0MS15,NaN,155922.72,NaN,NaN,NaN,NaN,98379.760,NaN,205096.840,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Q9Y6X5,172245.580,75492.77,234866.640,236185.100,116165.43,NaN,101202.790,97747.30,91552.900,138108.81,100396.54,125271.04,120986.15,192245.06,132578.12,250603.88,NaN,193052.60,8,8
Q9Y6X8,NaN,NaN,75900.516,73050.945,NaN,140158.40,114633.710,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,0
Q9Y6X9,84686.875,77121.77,83860.055,73670.620,75815.50,NaN,56846.727,71174.01,60096.023,78984.96,NaN,NaN,92552.98,78209.92,64695.96,91534.67,NaN,76899.05,8,6


In [10]:
df_filtererd = df_pivot[(df_pivot['T1D_count'] >= 5) | (df_pivot['Ctrl_count'] >= 5) ]
df_filtererd = df_filtererd.drop(columns=['T1D_count', 'Ctrl_count'])
df_filtererd

Sample ID,sample_01,sample_02,sample_03,sample_05,sample_06,sample_07,sample_08,sample_09,sample_10,sample_11,sample_12,sample_13,sample_15,sample_16,sample_17,sample_18,sample_19,sample_20
Protein ID,,,,,,,,,,,,,,,,,,
A0A0B4J1X5,338273.000,324902.62,NaN,351166.20,392492.70,455752.200,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
A0A0B4J2D5,1097570.900,938653.94,834068.060,916528.25,1015785.56,776846.750,744368.300,1195926.600,645568.700,866892.900,771991.06,610990.200,919461.000,553471.06,1221946.40,926734.900,620681.100,779474.44
A0A2R8Y4L2,2027513.400,1397669.20,1608827.200,2397105.20,1833632.40,2976784.800,4173473.200,1779404.800,1280212.500,1330438.400,1665231.20,1272206.500,1666470.600,1856255.40,1708183.60,1750802.200,1839920.400,1866433.40
A0A2R8Y619,3105330.500,4088374.00,4075224.500,3303696.80,3682206.00,NaN,3863942.000,2644634.500,2900838.500,NaN,3106510.80,NaN,NaN,2596980.80,NaN,1768318.900,NaN,2899720.80
A0AV96,469312.880,588717.06,644534.000,358684.62,506704.70,581558.800,650155.200,560454.750,641581.900,503776.780,426378.72,440250.280,508182.200,516928.44,550557.25,583201.060,511340.800,527829.20
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Q9Y6W3,129109.790,130161.87,105610.734,114774.39,106326.15,122152.445,124349.730,110513.086,112780.880,108915.520,133107.73,112018.945,116111.695,97164.42,132764.90,122303.484,124381.195,NaN
Q9Y6W5,168962.110,165205.92,98795.080,214356.78,162304.62,NaN,167478.160,133804.610,NaN,122171.734,165534.11,116923.125,NaN,196062.66,NaN,215397.110,247590.020,197430.73
Q9Y6X5,172245.580,75492.77,234866.640,236185.10,116165.43,NaN,101202.790,97747.300,91552.900,138108.810,100396.54,125271.040,120986.150,192245.06,132578.12,250603.880,NaN,193052.60


In [17]:
df_log = np.log2(df_filtererd)
sample_median = df_log.median(axis=0)
global_median = sample_median.median()
#sample_median, global_median
df_norm = df_log.subtract(sample_median, axis=1) + global_median

In [33]:
df_imputed_mnar = df_norm.fillna(df_norm.min().min() - 1.0)


In [35]:
impute = KNNImputer(n_neighbors=3, weights='distance')
imputed_array = impute.fit_transform(df_norm.T)
df_imputed_KNN = pd.DataFrame(imputed_array, index=df_norm.T.index, columns=df_norm.T.columns).T

In [36]:
long_mnar = df_imputed_mnar.reset_index().melt(id_vars='Protein ID', var_name='Sample ID', value_name='Intensity')
long_knn = df_imputed_KNN.reset_index().melt(id_vars='Protein ID', var_name='Sample ID', value_name='Intensity')

with sqlite3.connect(db_path) as conn:
    long_mnar.to_sql('imputed_MNAR', conn, if_exists='replace', index=False)
    long_knn.to_sql('imputed_KNN', conn, if_exists='replace', index=False)